# 08.01 · Persistence: checkpointers, threads, and durability

**Module 08 · LangGraph Production Features** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Module 07 (graphs, `InMemorySaver`, `get_state_history`); Lesson 03.03 (short-term memory). |
| 🔑 Keys | none |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Read** a checkpoint (`StateSnapshot`): `values`, `next`, `tasks`, `metadata`, `config`, `parent_config`.
2. **Persist** threads in SQLite so they survive a restart, and say when Postgres is the right next step.
3. **Measure** the three durability modes (`"exit"`, `"async"`, `"sync"`) by killing a run on purpose.
4. **Keep** a checkpoint database small (`delete_thread`, trimming old checkpoints) and **serialize** state safely.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) | Threads, checkpoints, `StateSnapshot`, `get_state`/`get_state_history`, serializers |
| [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers) | `InMemorySaver`, `SqliteSaver`/`AsyncSqliteSaver`, `PostgresSaver`, deleting threads |
| [Durable execution](https://docs.langchain.com/oss/python/langgraph/durable-execution) | The `durability` argument and what each mode guarantees |
| [Add memory](https://docs.langchain.com/oss/python/langgraph/add-memory) | Short-term memory = checkpointer + `thread_id` (recap of Module 03) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langgraph-checkpoint-sqlite>=3.1.1,<4"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langgraph-checkpoint", "langgraph-checkpoint-sqlite", "aiosqlite",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

> Everything in this lesson is 🧪 offline: no model is called. `GROQ_API_KEY` is only shown for consistency.

---
## Part 1 · Anatomy of a checkpoint

A **checkpointer** saves the graph's state after every **super-step** (one round of nodes) under a **thread** (`thread_id`). Each saved state is a **checkpoint**. A tiny two-node graph is enough to see all of it.

In [ ]:
# 🧪 Offline: a 2-node graph with an in-memory checkpointer.
import operator
from typing import Annotated

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


class Counter(TypedDict, total=False):
    x: int
    log: Annotated[list[str], operator.add]


def double(state: Counter) -> dict:
    return {"x": state["x"] * 2, "log": ["double"]}


def add_ten(state: Counter) -> dict:
    return {"x": state["x"] + 10, "log": ["add_ten"]}


builder = StateGraph(Counter)
builder.add_node("double", double)
builder.add_node("add_ten", add_ten)
builder.add_edge(START, "double")
builder.add_edge("double", "add_ten")
builder.add_edge("add_ten", END)
graph = builder.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "anatomy-1"}}
print(graph.invoke({"x": 1}, cfg))

**You should now see** `{'x': 12, 'log': ['double', 'add_ten']}`: (1 × 2) + 10.

In [ ]:
# 🧪 Offline: the newest checkpoint of the thread.
snap = graph.get_state(cfg)
print("values       :", snap.values)
print("next         :", snap.next)
print("tasks        :", snap.tasks)
print("metadata     :", {k: snap.metadata[k] for k in ("source", "step")})
print("checkpoint_id:", snap.config["configurable"]["checkpoint_id"][:13], "...")
print("parent       :", snap.parent_config["configurable"]["checkpoint_id"][:13], "...")

**You should now see** the final `values`, `next: ()` and `tasks: ()` (nothing left to run), `metadata` with `source: 'loop'` and `step: 2`, and two different checkpoint ids: this checkpoint and its **parent**.

| Field | Meaning |
|---|---|
| `values` | The state at this checkpoint |
| `next` | Nodes that will run next (empty = finished; non-empty = paused or crashed mid-run) |
| `tasks` | Those pending tasks, with errors and **interrupts** (Lesson 08.03) |
| `metadata.source` | `input` (your input), `loop` (after a super-step), `update` (`update_state`), `fork` |
| `metadata.step` | `-1` for the input, then `0, 1, 2, …` |
| `config` / `parent_config` | Ids that link checkpoints into a chain (a tree once you fork, Lesson 08.04) |

In [ ]:
# 🧪 Offline: the full history, oldest first.
for s in reversed(list(graph.get_state_history(cfg))):
    print(f"step {s.metadata['step']:>2}  source={s.metadata['source']:<5}  next={str(s.next):<13} values={s.values}")

**You should now see** 4 checkpoints: step `-1` (`source=input`, `next=('__start__',)`, only `log: []`), step `0` (`next=('double',)`, `x=1`), step `1` (`next=('add_ten',)`, `x=2`) and step `2` (`next=()`, `x=12`).

### What just happened?

One `invoke` wrote **one checkpoint per super-step**. Because each one knows its parent, LangGraph can resume a crashed run (from the newest), pause for a human (Lesson 08.03) and rewind or fork (Lesson 08.04). The `thread_id` is the key: a new id = a fresh conversation; the same id = continue.

---
## Part 2 · SQLite: survive a restart

`InMemorySaver` loses everything when the kernel restarts. `SqliteSaver` writes the same checkpoints to a file. `mastery.graphs.persistence.sqlite_checkpointer(path)` creates the folder, opens the connection (`check_same_thread=False`, which `SqliteSaver` guards with its own lock) and calls `setup()`.

In [ ]:
# 🧪 Offline: run on SQLite, close the connection ("restart"), open a NEW saver on the same file.
from mastery.graphs.persistence import count_checkpoints, file_size, sqlite_checkpointer
from mastery.memory import close

OUT = REPO_ROOT / "data" / "outputs" / "m08"
DB = OUT / "08-01-checkpoints.sqlite"
for suffix in ("", "-wal", "-shm"):                                  # a fresh file every run
    DB.with_name(DB.name + suffix).unlink(missing_ok=True)

saver = sqlite_checkpointer(DB)
builder.compile(checkpointer=saver).invoke({"x": 5}, {"configurable": {"thread_id": "sqlite-1"}})
close(saver)                                                          # the "process" ends

saver = sqlite_checkpointer(DB)                                       # a new process opens the file
graph_sql = builder.compile(checkpointer=saver)
print("after restart:", graph_sql.get_state({"configurable": {"thread_id": "sqlite-1"}}).values)
print("checkpoints  :", count_checkpoints(saver, "sqlite-1"), "| file:", file_size(DB))

**You should now see** `after restart: {'x': 20, 'log': ['double', 'add_ten']}` and `checkpoints  : 4` with a size in KB. The state came from the file, not from memory.

### Which checkpointer when?

| Checkpointer | Package | Use it for |
|---|---|---|
| `InMemorySaver` | `langgraph` | Notebooks, tests. Lost on restart |
| `SqliteSaver` / `AsyncSqliteSaver` | `langgraph-checkpoint-sqlite` | One machine, one process: local apps, this course |
| `PostgresSaver` / `AsyncPostgresSaver` | `langgraph-checkpoint-postgres` | Several workers or machines, real deployments (not installed in this course; needs a Postgres server) |
| LangSmith Deployment / `langgraph dev` | built in | The server manages persistence for you: **don't pass a checkpointer** to a graph you deploy |

> **Async graphs need the async saver.** `SqliteSaver` raises `NotImplementedError` from `ainvoke`/`astream`. Use `AsyncSqliteSaver` (helper: `await async_sqlite_checkpointer(path)`), which Lesson 08.02 needs for node timeouts.

---
## Part 3 · Durability: what survives a crash?

`invoke(..., durability=...)` decides **when** checkpoints are written:

| Mode | Writes | Trade-off |
|---|---|---|
| `"exit"` | Only when the run ends (finishes, errors or pauses) | Fastest; a crash loses the whole run |
| `"async"` (default) | In the background while the next step runs | Fast; a crash can lose the latest steps |
| `"sync"` | Before the next step starts | Slowest; nothing completed is lost |

Let's measure it. `mastery.graphs.research.run_and_kill` runs **Project 08's research graph** in a child Python process (offline models, SQLite checkpointer), with a slow document search, and kills the process hard (like a power cut) while it searches.

In [ ]:
# 🧪 Offline: kill a run mid-search in each durability mode (~10 s, three child processes).
from mastery.graphs.research import run_and_kill

QUESTION = "How do I keep the battery healthy in winter?"
crash_dbs = {}
for mode in ("sync", "async", "exit"):
    crash_db = OUT / f"08-01-crash-{mode}.sqlite"
    for suffix in ("", "-wal", "-shm"):
        crash_db.with_name(crash_db.name + suffix).unlink(missing_ok=True)
    lines = run_and_kill(crash_db, "crash", QUESTION, durability=mode, slow=3)
    crash_dbs[mode] = crash_db
    print(f"{mode:<5}:", " | ".join(line.strip() for line in lines[1:]))

**You should now see** three identical lines per mode: `STEP clarify_question | STEP plan | ...searching docs for '...' (3s) | 💥 killed`. The process died at the same place each time. What's in the files?

In [ ]:
# 🧪 Offline: open each file in THIS process and look at the newest root checkpoint.
for mode, crash_db in crash_dbs.items():
    s = sqlite_checkpointer(crash_db)
    tup = s.get_tuple({"configurable": {"thread_id": "crash"}})
    if tup is None:
        print(f"{mode:<5}: nothing saved")
    else:
        meta = tup.metadata
        print(f"{mode:<5}: newest step={meta['step']} source={meta['source']}  ({count_checkpoints(s, 'crash')} checkpoints incl. subgraph)")
    close(s)

**You should now see** (measured on langgraph 1.2.14):

- `sync : newest step=2 source=loop`: `clarify_question` and `plan` are saved; a resume starts at `research`.
- `async: ...` **fewer** steps, measured `step=-1 source=input` (only the input): the background writes hadn't landed. This line depends on timing and can differ on your machine.
- `exit : nothing saved`: the run never ended, so nothing was written.

### What just happened?

Same crash, three outcomes. `"sync"` costs a little latency per step and buys you "no completed step is ever lost". Use it when steps are slow or expensive (LLM calls, paid APIs, human approvals); keep `"async"` for cheap, fast graphs. Lesson 08.02 resumes the `sync` run.

---
## Part 4 · Keep the database small

Every super-step adds a checkpoint, forever. A chat thread with 200 turns × 5 steps = 1,000 checkpoints. Two tools:

- `saver.delete_thread(thread_id)`: remove a thread completely (a user deletes a conversation; GDPR).
- `trim_thread(saver, thread_id)` (course helper): keep only the **newest** checkpoint. The thread continues normally, but its history (time travel) is gone.

> `saver.prune(thread_ids, strategy="keep_latest")` is in the base checkpointer API, but `SqliteSaver` 3.1.1 raises `NotImplementedError` for it (measured 2026-10-12). That's why the course has `trim_thread`; check the [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers) page for newer releases.

In [ ]:
# 🧪 Offline: 30 short runs on one thread, then trim and delete.
from mastery.graphs.persistence import trim_thread

long_cfg = {"configurable": {"thread_id": "long"}}
for i in range(30):
    graph_sql.invoke({"x": i}, long_cfg)
print("before trim :", count_checkpoints(saver, "long"), "checkpoints |", file_size(DB))
print("trimmed     :", trim_thread(saver, "long"))
print("after trim  :", count_checkpoints(saver, "long"), "| state still there:", graph_sql.get_state(long_cfg).values["x"])
saver.delete_thread("long")
print("after delete:", count_checkpoints(saver, "long"), "| other thread kept:", count_checkpoints(saver, "sqlite-1"))

**You should now see** `before trim : 120 checkpoints | 2.2 MB` (30 runs × 4; the `log` list grows on every run, so later checkpoints are bigger), `trimmed     : 119`, `after trim  : 1 | state still there: 68` (the last run: 29 × 2 + 10), then `after delete: 0 | other thread kept: 4`.

The **file** doesn't shrink by itself: SQLite reuses the free pages. Run `VACUUM` (`saver.conn.execute("VACUUM")`) during maintenance if you need the disk space back.

---
## Part 5 · What can go into state? Serialization

Checkpointers store state with a **serializer** (`JsonPlusSerializer`). JSON-like values (`str`, `int`, `list`, `dict`), LangChain messages and a few standard types are always fine. **Your own classes** (Pydantic models, dataclasses) are stored with their module and class name, and on reading back LangGraph only rebuilds types you **allow**. With an allow-list, anything else is **blocked**.

In [ ]:
# 🧪 Offline: a Pydantic object in state, read back with and without an allow-list.
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
from pydantic import BaseModel


class Ticket(BaseModel):
    title: str
    priority: int


serde_strict = JsonPlusSerializer(allowed_msgpack_modules=[("mastery.graphs.research", "Plan")])   # Ticket NOT listed
serde_allowed = JsonPlusSerializer(allowed_msgpack_modules=[(Ticket.__module__, "Ticket")])

blob = serde_strict.dumps_typed(Ticket(title="Battery drains", priority=2))
print("stored as      :", blob[0])
for name, serde in (("not allowed", serde_strict), ("allowed", serde_allowed)):
    back = serde.loads_typed(blob)
    print(f"{name:<15}: {type(back).__name__} -> {back!r}")

**You should now see** `stored as      : msgpack`, a log line `Blocked deserialization of __main__.Ticket - not in allowed_msgpack_modules ...`, then `not allowed` gives back a plain `dict` and `allowed` gives back a real `Ticket`.

A blocked object silently turns into something else: code that does `state["ticket"].priority` breaks **after a restart**, not in your first test. The Module 07 helpers pass an allow-list for the course models (`mastery.graphs.state.CHECKPOINT_TYPES`).

**The course rule (D-030): keep checkpointed state plain** (`str`, numbers, lists, dicts, messages). Validate with Pydantic **inside** a node, then store `model.model_dump()`. A deployed server (`langgraph dev`, LangSmith Deployment) has no allow-list setting, so it warns "Deserializing unregistered type" for your classes; dicts work everywhere.

In [ ]:
# 🧪 Offline: close the connections so the files can be deleted (Windows locks open files).
close(saver)
print("closed; files in", OUT.relative_to(REPO_ROOT), "->", sorted(p.name for p in OUT.glob("08-01-*.sqlite")))

**You should now see** `closed; files in data/outputs/m08 -> [...]` with the four `.sqlite` files. `data/outputs/` is git-ignored; delete the folder whenever you like.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| A second run "remembers" the first | Same `thread_id` | New id per conversation (`uuid.uuid4().hex`) |
| `ValueError: Checkpointer requires ... thread_id` | No `configurable.thread_id` in the config | Pass `{"configurable": {"thread_id": ...}}` |
| `NotImplementedError` from `ainvoke` | `SqliteSaver` in an async graph | `AsyncSqliteSaver` (`await async_sqlite_checkpointer(path)`) |
| `ProgrammingError: SQLite objects created in a thread ...` | `sqlite3.connect` without `check_same_thread=False` | Use the helper, or pass it yourself |
| `PermissionError: [WinError 32]` deleting the `.sqlite` | The connection is still open (Windows locks files) | `close(saver)` first, or restart the kernel |
| `Deserializing unregistered type ...` / blocked type | Your own class in state | Store `model_dump()` dicts, or add the type to the allow-list |
| Lost the last steps after a crash | Default `durability="async"` | `durability="sync"` for slow or costly steps |

## ⚠️ Production notes

> ⚠️ **Production note: one SQLite file = one machine.**
> SQLite is great for a single process. Several workers writing at once need `PostgresSaver` (or the managed persistence of LangSmith Deployment). Don't put a SQLite file on a network drive.

> ⚠️ **Production note: plan retention from day one.**
> Decide how long threads live, delete them on request (`delete_thread`), trim long ones, and remember that checkpoints contain user data (prompts, answers, PII): treat the file like a database of personal data.

## 🏋️ Exercises

1. **(Easy)** In Part 1, run `graph.invoke({"x": 3}, cfg)` again on the **same** thread. How many checkpoints does the history have now, and what is `log`?
2. **(Medium)** Time `graph_sql.invoke` 50 times with `durability="sync"` and 50 times with `"exit"` (`time.perf_counter`). How big is the difference for this tiny graph?
3. **(Medium)** Put a `datetime.date` and a `set` into the `Counter` state (add fields). Do they come back from SQLite unchanged?
4. **(Stretch)** Write `export_thread(saver, thread_id) -> list[dict]` that dumps every checkpoint's `step`, `source` and `values` to JSON (e.g. for a data-access request).

<details>
<summary><b>Show solutions</b></summary>

**1.** 8 checkpoints (4 per invoke), and `log` is `['double', 'add_ten', 'double', 'add_ten']`: `x` is overwritten by the new input, but `log` has a reducer (`operator.add`), so it grows. That's why chat history keeps growing on one thread.

**2.** A few milliseconds per run at most here; the difference matters when each write is a network round trip (Postgres) and steps are fast. For LLM steps (seconds), `"sync"` is noise.

**3.** Yes: `JsonPlusSerializer` handles common standard-library types (dates, sets, UUIDs, Decimals). Your **own** classes are the ones that need the allow-list.

**4.**
```python
import json
def export_thread(graph, thread_id):
    cfg = {"configurable": {"thread_id": thread_id}}
    return [{"step": s.metadata["step"], "source": s.metadata["source"], "values": s.values}
            for s in graph.get_state_history(cfg)]
print(json.dumps(export_thread(graph, "anatomy-1"), indent=1, default=str)[:400])
```
</details>

## ✅ Checkpoint

- [ ] I can name every field of a `StateSnapshot` and what `next` tells me.
- [ ] I persisted a thread in SQLite and read it back from a new saver.
- [ ] I can explain `"exit"` / `"async"` / `"sync"` with the measured crash results.
- [ ] I know how to delete and trim threads, and why state should stay plain.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers) and note in [`docs/research-log.md`](../../docs/research-log.md): which checkpointer packages exist today (Postgres, Redis, others), whether `SqliteSaver.prune` is implemented in the newest `langgraph-checkpoint-sqlite` (this lesson measured 3.1.1), and what the docs say about **encrypting** checkpoints (`EncryptedSerializer`).

## Summary

- A checkpoint per super-step, per thread; `get_state` = newest, `get_state_history` = all, linked by parent ids.
- SQLite for one machine, Postgres for many, nothing when the server manages persistence.
- `durability="sync"` keeps every completed step through a crash; `"async"` and `"exit"` trade safety for speed.
- Delete and trim threads; store plain data (`model_dump()`), not custom classes.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 08.02 · Durable execution and fault tolerance** → [`08-02-durable-execution-and-fault-tolerance.ipynb`](08-02-durable-execution-and-fault-tolerance.ipynb).

[Module 08 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)